In [2]:
import json
from pathlib import Path
from yarn_utils import YARNGraph
import itertools
import networkx as nx

# Load Data

In [3]:
# FOLDER_PATH = "annotations/FRACAS_12032026/"
# FILE = "105h.yarn.json"

# FOLDER_PATH = "annotations/"
# FILE = "1.yarn.json"

# FOLDER_PATH = "annotations/FRACAS_1premise_yesno/"
# FILE = "106h.yarn.json"

FOLDER_PATH = "annotations/FRACAS_1premise_yesno/"
FILE = "17p.yarn.json"

In [4]:
with open(FOLDER_PATH + FILE) as f:
    yarn_graph_json = json.load(f)

yarn_graph = YARNGraph(yarn_graph_json)
yarn_grew = yarn_graph.grew()

In [5]:
with open('output.json', 'w') as f:
    json.dump(yarn_grew, f)

In [6]:
yarn_grew

{'nodes': {'vm1': {'concept': 'man', 'type': 'V', 'var': 'vm1'},
  'vi1': {'concept': 'irish', 'type': 'V', 'var': 'vi1'},
  'vp1': {'concept': 'prize', 'type': 'V', 'var': 'vp1'},
  'vn1': {'concept': 'Nobel Prize', 'type': 'V', 'var': 'vn1'},
  'vl1': {'concept': 'literature', 'type': 'V', 'var': 'vl1'},
  'vw1': {'pred': 'win-01', 'type': 'V', 'var': 'vw1'},
  's1': {'event': 's1', 'var': 's1', 'type': 'S'},
  's1-temp': {'type': 'F', 'feat': 'temp', 'var': 's1-temp'},
  's1-quant': {'type': 'F', 'feat': 'quant', 'var': 's1-quant'},
  's1-num': {'type': 'F', 'feat': 'num', 'var': 's1-num'},
  's1-def': {'type': 'F', 'feat': 'def', 'var': 's1-def'},
  's1-aspect': {'type': 'F', 'feat': 'aspect', 'var': 's1-aspect'},
  'e1': {'rel': 'name', 'type': 'E', 'var': 'e1'},
  'e2': {'rel': 'mod', 'type': 'E', 'var': 'e2'},
  'e3': {'rel': 'mod', 'type': 'E', 'var': 'e3'},
  'e4': {'rel': 'ARG0', 'type': 'E', 'var': 'e4'},
  'e5': {'rel': 'ARG1', 'type': 'E', 'var': 'e5'},
  'l1': {'value': '

# Preprocessing

In [7]:
# from grewpy import Graph, GRS

# grs_path = "grs/main.grs"
# grs = GRS(grs_path)
# yarn_grew = grs.apply(Graph(yarn_grew), strat='main')

In [8]:
# convert temp H edges to L edges
# domain, mod, be
# have
# potentially add existential quantification if constants idea is not good

# Build F and R

In [9]:
def build_F(yarn_grew_graph, id2var, variables):

    variables = set() #move?
    def fresh_variable(base=None):
        if base is None:
            base = 'e'
        i = 0

        while True:
            variable = base if i == 0 else f"{base}{i}"
            if variable not in variables:
                variables.add(variable)
                break
            i += 1
        return variable

    F = []

    for node, feats in yarn_grew_graph['nodes'].items():

        # S node quantifcation
        # includes V -> S (C edges)
        # no S -> S (D edges) for now 
        if feats['type'] == 'S':

            id2var[node] = feats['var']
            variables.add(feats['var'])

            for edge1 in yarn_grew_graph['edges']:
                if edge1['tar'] == node:
                    src = edge1['src']

                    if yarn_grew_graph['nodes'][src]['type'] == 'C':
                        for edge2 in yarn_grew_graph['edges']:
                            if edge2['tar'] == src:
                                incoming = edge2['src']
                else:
                    incoming = None

            F.append({
                            'id':node,
                            'incoming': incoming,
                            'outgoing':None,
                            'type':"∃_s",
                            'variable': id2var[node],
                            'tar_label': 'S', # change to 'label'
                        })
            
        # quantification
        if (feats['type'] == "L" or feats['type'] == 'H') and feats['feat'] in ['quant', 'temp']:
            for edge in yarn_grew_graph['edges']:
                if edge['src'] == node:
                    tar = edge['tar']

                    if yarn_grew_graph['nodes'][tar]['type'] == 'V':
                        
                        edge_label = feats['value'] if feats['value'] else feats['feat'] # accepts unlabeled temp/quant edges for now

                        if 'pred' in yarn_grew_graph['nodes'][tar]:
                            tar_label = yarn_grew_graph['nodes'][tar]['pred']
                        else:
                            tar_label = yarn_grew_graph['nodes'][tar]['concept']

                        if tar not in id2var:
                            variable = fresh_variable(base=tar_label[0])
                            id2var[tar] = variable
                        else:
                            raise AssertionError(f"Double quantification. Variable for {tar} already exists in id2var.")
                        
                        F.append({
                            'id':tar,
                            'incoming':node,
                            'outgoing':None,
                            'type':"Q_"+edge_label,
                            'variable': id2var[tar] if feats['feat'] in ['quant', 'temp'] else None,
                            'tar_label':tar_label, # change to 'label'
                        })
        
        # negation, modality, aspect
        if (feats['type'] == "L" or feats['type'] == 'H') and feats['feat'] in ['neg', 'modal', 'aspect']:
            for edge1 in yarn_grew_graph['edges']:
                if edge1['src'] == node:
                    tar = edge1['tar']

                    if yarn_grew_graph['nodes'][tar]['type'] in ['V', 'L', 'H']:
                        
                        edge_label = feats['value'] if feats['value'] else feats['feat']

                        for edge2 in yarn_grew_graph['edges']:
                            if edge2['tar'] == node:
                                src = edge2['src']

                        F.append({
                            'id':node,
                            'incoming':src,
                            'outgoing':tar,
                            'type':"Q_"+edge_label,
                            'variable': None,
                            'tar_label': None, # change to 'label'
                        })
        
        # Any none-quantified V node is a constant
    for node, feats in yarn_grew_graph['nodes'].items():
        if feats['type'] == 'V' and node not in id2var:
            if 'pred' in feats:
                id2var[node] = feats['pred'].upper()
            else:
                id2var[node] = feats['concept'].upper()

    F = [f for f in F if f['type'] not in ['Q_perfective', 'Q_state']]

    return F

In [10]:
yarn_grew

{'nodes': {'vm1': {'concept': 'man', 'type': 'V', 'var': 'vm1'},
  'vi1': {'concept': 'irish', 'type': 'V', 'var': 'vi1'},
  'vp1': {'concept': 'prize', 'type': 'V', 'var': 'vp1'},
  'vn1': {'concept': 'Nobel Prize', 'type': 'V', 'var': 'vn1'},
  'vl1': {'concept': 'literature', 'type': 'V', 'var': 'vl1'},
  'vw1': {'pred': 'win-01', 'type': 'V', 'var': 'vw1'},
  's1': {'event': 's1', 'var': 's1', 'type': 'S'},
  's1-temp': {'type': 'F', 'feat': 'temp', 'var': 's1-temp'},
  's1-quant': {'type': 'F', 'feat': 'quant', 'var': 's1-quant'},
  's1-num': {'type': 'F', 'feat': 'num', 'var': 's1-num'},
  's1-def': {'type': 'F', 'feat': 'def', 'var': 's1-def'},
  's1-aspect': {'type': 'F', 'feat': 'aspect', 'var': 's1-aspect'},
  'e1': {'rel': 'name', 'type': 'E', 'var': 'e1'},
  'e2': {'rel': 'mod', 'type': 'E', 'var': 'e2'},
  'e3': {'rel': 'mod', 'type': 'E', 'var': 'e3'},
  'e4': {'rel': 'ARG0', 'type': 'E', 'var': 'e4'},
  'e5': {'rel': 'ARG1', 'type': 'E', 'var': 'e5'},
  'l1': {'value': '

In [11]:
F= build_F(yarn_grew, id2var={}, variables=set())
F

[{'id': 's1',
  'incoming': None,
  'outgoing': None,
  'type': '∃_s',
  'variable': 's1',
  'tar_label': 'S'},
 {'id': 'vw1',
  'incoming': 'l1',
  'outgoing': None,
  'type': 'Q_past',
  'variable': 'w',
  'tar_label': 'win-01'},
 {'id': 'vm1',
  'incoming': 'l2',
  'outgoing': None,
  'type': 'Q_exists',
  'variable': 'm',
  'tar_label': 'man'},
 {'id': 'vp1',
  'incoming': 'l3',
  'outgoing': None,
  'type': 'Q_exists',
  'variable': 'p',
  'tar_label': 'prize'}]

In [12]:
def build_R(yarn_grew_graph, id2var):

    R = {}
    for node, feats in yarn_grew_graph['nodes'].items():
        if feats['type'] == "E":
            edge_label = feats['rel']

            for edge1 in yarn_grew_graph['edges']:
                if edge1['tar'] == node:

                    src = edge1['src']

                    for edge2 in yarn_grew_graph['edges']:
                        if edge2['src'] == node:
                            tar = edge2['tar']

                            if id2var[src].isupper(): # if constant
                                if tar not in R:
                                    R[tar] = [(edge_label, src, tar)]
                                else:
                                    R[tar].append((edge_label, src, tar))
                            else:
                                if src not in R:
                                    R[src] = [(edge_label, src, tar)]
                                else:
                                    R[src].append((edge_label, src, tar))
        
        if feats['type'] == "L" and feats['feat'] == 'num' and feats['value'] == 'plural': 
            for edge1 in yarn_grew_graph['edges']:
                if edge1['src'] == node:
                    tar = edge1['tar']

                    if yarn_grew_graph['nodes'][tar]['type'] == 'V':
                        
                        if tar not in R:
                            R[tar] = [('plural', tar)] 
                        else:
                            R[tar].append(('plural', tar))
        
        if feats['type'] == "L" and feats['feat'] == 'def': 
            for edge1 in yarn_grew_graph['edges']:
                if edge1['src'] == node:
                    tar = edge1['tar']

                    if yarn_grew_graph['nodes'][tar]['type'] == 'V':
                        
                        if tar not in R:
                            R[tar] = [('C', tar)]
                        else:
                            R[tar].append(('C', tar))

        if feats['type'] == "C":
            edge_label = feats['rel']

            for edge1 in yarn_grew_graph['edges']:
                if edge1['tar'] == node:
                    src = edge1['src']

                    for edge2 in yarn_grew_graph['edges']:
                        if edge2['src'] == node:
                            tar = edge2['tar']
                    
                            if tar not in R:
                                R[tar] = [(edge_label, src, tar)]
                            else:
                                R[tar].append((edge_label, src, tar))

        # S node modifiers
        # update as you annotate if you find trickier cases
            
        if feats['type'] == "L" and feats['feat'] in ['manner', 'loc', 'dir', 'duration', 'mod', 'freq']: 
            for edge1 in yarn_grew_graph['edges']:
                if edge1['src'] == node:
                    tar = edge1['tar']

                    if yarn_grew_graph['nodes'][tar]['type'] == 'V':
                        
                        edge_label = feats['value'] if feats['value'] else feats['feat']

                        if 'pred' in yarn_grew_graph['nodes'][tar]:
                            tar_label = yarn_grew_graph['nodes'][tar]['pred']
                        else:
                            tar_label = yarn_grew_graph['nodes'][tar]['concept']

                        for edge2 in yarn_grew_graph['edges']:
                            if edge2['tar'] == node:
                                src = edge2['src'].split('-')[0]
                            
                                if src not in R:
                                    R[src] = [(edge_label, src, tar)]
                                else:
                                    R[src].append((edge_label, src, tar))
    
    return R

# Create the Forest

In [13]:
def build_scope_forest(F, R, id2var):

    forest = {'nodes':{}, 'edges':[]}

    for i, f in enumerate(F):
        forest['nodes'][i] = {
                                'id': f['id'],
                                'incoming': f['incoming'],
                                'outgoing': f['outgoing'],
                                'type': f['type'],
                                'variable': f['variable'],
                                'tar_label': f['tar_label'],
                                'relations': [
                                    f"{rel[0]}({id2var[rel[1]]},{id2var[rel[2]]})" if len(rel) == 3
                                    else f"{rel[0]}({id2var[rel[1]]})"
                                    for rel in R[f['id']]
                                ] if f['id'] in R else [],
                            }


    for k1, v1 in forest['nodes'].items(): # encode specified scope
        for k2, v2 in forest['nodes'].items():
            if v1['id'] == v2['incoming']:
                forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})
            if v1['id'] == v2['outgoing']:
                forest['edges'].append({'src':k2, 'rel':'', 'tar':k1})
            if v1['outgoing'] and v2['incoming'] and v1['outgoing'] == v2['incoming']: # not sure this is smart
                forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})

    return forest

## Add the Participant before Event constraint

In [14]:
# Predicates are introduced after their arguments (E relations only)
# C relations are encoded already in the Forest building forest

def add_participants_before_event_principle(forest, yarn_grew_graph, R):

    for _, rels in R.items():
        for rel in rels:
            if len(rel) == 3:
                src = rel[1]
                tar = rel[2]

            for k1, v1 in forest['nodes'].items():
                for k2, v2 in forest['nodes'].items():
                    if v1['id'] == src and v2['id'] == tar and \
                        yarn_grew_graph['nodes'][src]['type'] == 'V' and \
                        yarn_grew_graph['nodes'][tar]['type'] == 'V':
                        
                        forest['edges'].append({'src':k2, 'rel':'', 'tar':k1})
    
    return forest

In [15]:
yarn_grew

{'nodes': {'vm1': {'concept': 'man', 'type': 'V', 'var': 'vm1'},
  'vi1': {'concept': 'irish', 'type': 'V', 'var': 'vi1'},
  'vp1': {'concept': 'prize', 'type': 'V', 'var': 'vp1'},
  'vn1': {'concept': 'Nobel Prize', 'type': 'V', 'var': 'vn1'},
  'vl1': {'concept': 'literature', 'type': 'V', 'var': 'vl1'},
  'vw1': {'pred': 'win-01', 'type': 'V', 'var': 'vw1'},
  's1': {'event': 's1', 'var': 's1', 'type': 'S'},
  's1-temp': {'type': 'F', 'feat': 'temp', 'var': 's1-temp'},
  's1-quant': {'type': 'F', 'feat': 'quant', 'var': 's1-quant'},
  's1-num': {'type': 'F', 'feat': 'num', 'var': 's1-num'},
  's1-def': {'type': 'F', 'feat': 'def', 'var': 's1-def'},
  's1-aspect': {'type': 'F', 'feat': 'aspect', 'var': 's1-aspect'},
  'e1': {'rel': 'name', 'type': 'E', 'var': 'e1'},
  'e2': {'rel': 'mod', 'type': 'E', 'var': 'e2'},
  'e3': {'rel': 'mod', 'type': 'E', 'var': 'e3'},
  'e4': {'rel': 'ARG0', 'type': 'E', 'var': 'e4'},
  'e5': {'rel': 'ARG1', 'type': 'E', 'var': 'e5'},
  'l1': {'value': '

In [16]:
def get_S_descendants(yarn_grew_graph):
    nodes = yarn_grew_graph["nodes"]
    edges = yarn_grew_graph["edges"]

    adj = {}
    for e in edges:
        adj.setdefault(e["src"], []).append(e["tar"])

    result = {}

    for node_id, node_data in nodes.items():
        if node_data.get("type") != "S":
            continue

        visited = set()
        stack = [node_id]
        reachable_V = set()

        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)

            current_type = nodes[current].get("type")

            if current_type in ["L", "H", "V"]:
                reachable_V.add(current)

            if current_type == "C":
                continue

            for neighbor in adj.get(current, []):
                if neighbor not in visited:
                    stack.append(neighbor)

        result[node_id] = list(reachable_V)

    return result

In [17]:
def add_s_node_scope(forest):

    s_descendants = get_S_descendants(yarn_grew)
    for k1, v1 in forest['nodes'].items():
        if v1['id'] in s_descendants:
            for k2, v2 in forest['nodes'].items():
                if v2['id'] in s_descendants[v1['id']]:
                    forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})
                    
    return forest

# Get All Possible Trees

In [18]:
def get_all_possible_trees(n):
    nodes = list(range(n))
    for seq in itertools.product(nodes, repeat=n-2):
        yield nx.from_prufer_sequence(seq)

In [19]:
def get_all_possible_rooted_directed_trees(forest):

    nodes = list(forest['nodes'].keys())
    n_nodes = len(nodes)

    all_possible_rooted_directed_tree_edges = []

    for tree in get_all_possible_trees(n_nodes):

        for root in nodes:

            visited = set([root])
            stack = [root]
            directed_edges = []

            while stack:
                current = stack.pop()

                for neighbor in tree.neighbors(current):
                    if neighbor not in visited:
                        visited.add(neighbor)
                        stack.append(neighbor)

                        directed_edges.append({'src': current,'tar': neighbor})

            all_possible_rooted_directed_tree_edges.append({'edges': directed_edges})
    
    return all_possible_rooted_directed_tree_edges

# Build T_all

In [20]:
# Gets the children of nodes that don't introduce variables
def get_H_children(graph):
    H_children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        if not graph['nodes'][src]['variable']:
            H_children_dict[src] = H_children_dict.get(src, []) + [tar]

    return H_children_dict

In [21]:
# Get all children
def get_children(graph):
    children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        children_dict[src] = children_dict.get(src, []) + [tar]
    return children_dict

# Extend children to descendants
def get_descendants(node, children_dict):
    descendants = []
    for child in children_dict.get(node, []):
        descendants.append(child)
        descendants.extend(get_descendants(child, children_dict))

    return descendants

def get_all_descendants(graph):
    children_dict = get_children(graph)
    descendants_dict = {}
    for node in children_dict:
        descendants_dict[node] = get_descendants(node, children_dict)

    return descendants_dict

In [22]:
# Constraint Checkers

def check_compatibility_of_scopes(tree, forest):
    descendants_tree = get_all_descendants(tree)
    descendants_forest = get_all_descendants(forest)

    for k,v in descendants_forest.items():
        for descendant in v:
            if k in descendants_tree:
                if descendant not in descendants_tree[k]:
                    return False
            else:
                return False
    return True

def check_locality_of_features(tree, forest):
    children_tree = get_children(tree)
    H_children_forest = get_H_children(forest)
    
    for k,v in H_children_forest.items():
        for child in v:
            if k in children_tree:
                if child not in children_tree[k]:
                    return False
            else:
                return False
    return True

In [23]:
# descendants_forest = get_all_descendants(forest)
# H_children_forest = get_H_children(forest)
# print(descendants_forest)
# print(H_children_forest)

In [24]:
def build_T_all(forest, valid_tree_edges):

    T_all = []
    for tree in valid_tree_edges:
        new_tree = forest.copy()
        new_tree['edges'] = tree['edges']
        T_all.append(new_tree)
    
    return T_all

## Reformat T_all

In [25]:
# Reformat to linear tree for easier interpretation

def reformat(graph):
    nodes = graph["nodes"]
    edges = graph["edges"]

    next_node = {}
    for e in edges:
        next_node[e["src"]] = e["tar"]

    all_nodes = set(nodes.keys())
    all_targets = {e["tar"] for e in edges}
    root = (all_nodes - all_targets).pop()

    def build(node_id):
        node_data = dict(nodes[node_id])

        if node_id in next_node:
            node_data["child"] = build(next_node[node_id])
        else:
            node_data["child"] = None

        return node_data

    return build(root)

# Interpretation

In [26]:
def conj(parts):
    parts = [p for p in parts if p and p.strip()]
    return " ∧ ".join(parts)


def wrap_quant(q, var, head, relations, body, connective="∧"):
    """
    Builds:
    Qx. ( head(x) ∧ {rels} {connective} {body} )
    """

    rel = conj(relations)
    head_part = f"{head}({var})"

    if rel:
        left = f"{head_part} ∧ {rel}"
    else:
        left = head_part

    return f"{q}{var}. ( {left}\n {connective} ({body}) )"

def clean_formula(formula):
    clean_formula = formula.replace(" ∧ ()", "")
    return clean_formula

In [27]:
def interpret(root, temp_variable):
    
    if root is None:
        return ""
    
    if root["type"] == "∃_s" or root["type"] == "Q_exists":
        return wrap_quant(
            "∃",
            root["variable"],
            root["tar_label"],
            root["relations"],
            interpret(root["child"], temp_variable),
            connective="∧"
        )
    
    if root["type"] == "Q_forall":
        return wrap_quant(
            "∀",
            root["variable"],
            root["tar_label"],
            root["relations"],
            interpret(root["child"], temp_variable),
            connective="→"
        )
    
    if root["type"] == "Q_present":
        temp = f"{root['variable']}_O_{temp_variable}"
        return wrap_quant(
                    "∃",
                    root["variable"],
                    root["tar_label"],
                    root["relations"] + [temp],
                    interpret(root["child"], temp_variable),
                    connective="∧"
        )

    if root["type"] == "Q_past":
        temp = f"{root['variable']}≺{temp_variable}"
        return wrap_quant(
                    "∃",
                    root["variable"],
                    root["tar_label"],
                    root["relations"] + [temp],
                    interpret(root["child"], temp_variable),
                    connective="∧"
        )
    
    if root["type"] == "Q_future":
        temp = f"{temp_variable}≺{root['variable']}"
        return wrap_quant(
                    "∃",
                    root["variable"],
                    root["tar_label"],
                    root["relations"] + [temp],
                    interpret(root["child"], temp_variable),
                    connective="∧"
        )
    
    if root["type"] == "Q_neg":
        return f"¬( {interpret(root['child'], temp_variable)} )"
    
    if root["type"] == "Q_possibility":
        return f"◇( {interpret(root['child'], temp_variable)} )"

    if root["type"] == "Q_necessity":
        return f"□( {interpret(root['child'], temp_variable)} )"

In [28]:
# for tree in T_all:
#     print(clean_formula(interpret(tree, 'NOW')))
#     print("---")

In [29]:
def yarn2fol(yarn_graphs):

    for path, yarn_graph_json in yarn_graphs:

        try:
            id2var = {}
            variables = set()
        
            yarn_graph = YARNGraph(yarn_graph_json)
            yarn_graph_grew = yarn_graph.grew()

            F = build_F(yarn_graph_grew, id2var, variables)
            R = build_R(yarn_graph_grew, id2var)
            forest = build_scope_forest(F, R, id2var)

            forest = add_participants_before_event_principle(forest)
            forest = add_s_node_scope(forest)

            all_possible_rooted_directed_tree_edges = get_all_possible_rooted_directed_trees(forest)
            
            valid_tree_edges = [tree for tree in all_possible_rooted_directed_tree_edges if check_locality_of_features(tree, forest)]
            valid_tree_edges = [tree for tree in valid_tree_edges if check_compatibility_of_scopes(tree, forest)]

            T_all = build_T_all(forest, valid_tree_edges)
            T_all = [reformat(tree) for tree in T_all]

            print(path)
            print(yarn_graph_json['meta']['type'], ':', yarn_graph_json['meta']['text'])
            for T in T_all:
                print(clean_formula(interpret(T, 'NOW')))
                print("---")

        except:
            print(path)

In [30]:
import re

def extract_number(path):

    match = re.match(r"(\d+)", path.name)
    return int(match.group(1)) if match else float("inf")


def load_yarn(input_path, recursive=False):
    if isinstance(input_path, (str, Path)):
        input_path = [input_path]

    all_files = []

    for path in input_path:
        path = Path(path)

        if path.is_file():
            if path.name.endswith(".yarn.json"):
                all_files.append(path)

        elif path.is_dir():
            files = path.rglob("*.yarn.json") if recursive else path.glob("*.yarn.json")
            all_files.extend(files)

        else:
            raise FileNotFoundError(f"{path} does not exist")

    all_files = sorted(all_files, key=extract_number)

    graphs = []
    for file_path in all_files:
        print(file_path)
        with open(file_path, "r", encoding="utf-8") as f:
            graph = json.load(f)
            if graph.get('labels'):  # safer
                graphs.append((file_path, graph))

    return graphs

In [57]:
import traceback
import multiprocessing as mp


TIMEOUT = 20


def process_one(path, yarn_graph_json, output_queue):
    try:
        id2var = {}
        variables = set()

        yarn_graph = YARNGraph(yarn_graph_json)
        yarn_graph_grew = yarn_graph.grew()

        F = build_F(yarn_graph_grew, id2var, variables)
        R = build_R(yarn_graph_grew, id2var)
        forest = build_scope_forest(F, R, id2var)

        forest = add_participants_before_event_principle(forest, yarn_graph_grew, R)
        forest = add_s_node_scope(forest)
        
        all_possible_rooted_directed_tree_edges = get_all_possible_rooted_directed_trees(forest)

        valid_tree_edges = [
            tree for tree in all_possible_rooted_directed_tree_edges
            if check_locality_of_features(tree, forest)
        ]
        valid_tree_edges = [
            tree for tree in valid_tree_edges
            if check_compatibility_of_scopes(tree, forest)
        ]

        T_all = build_T_all(forest, valid_tree_edges)
        T_all = [reformat(tree) for tree in T_all]

        results = []
        for T in T_all:
            results.append(clean_formula(interpret(T, 'NOW')))

        output_queue.put({
            "path": str(path),
            "meta": yarn_graph_json.get("meta", {}),
            "results": results,
            "error": None
        })

    except Exception as e:
        output_queue.put({
            "path": str(path),
            "meta": yarn_graph_json.get("meta", {}),
            "results": None,
            "error": traceback.format_exc()
        })

def yarn2fol(yarn_graphs):
    for path, yarn_graph_json in yarn_graphs:

        print("\nProcessing:", path)

        output_queue = mp.Queue()
        p = mp.Process(target=process_one, args=(path, yarn_graph_json, output_queue))

        p.start()
        p.join(TIMEOUT)

        if p.is_alive():
            p.terminate()
            p.join()

            print(path)
            print("TIMEOUT after", TIMEOUT, "seconds")
            continue

        if output_queue.empty():
            print(path)
            print("No output returned")
            continue

        result = output_queue.get()

        if result["error"]:
            print(path)
            print("ERROR:")
            print(result["error"])
            continue

        print(result["path"])
        print(result["meta"].get("type"), ":", result["meta"].get("text"))

        for r in result["results"]:
            print(r)
            print("---")

In [58]:
corpus = load_yarn(FOLDER_PATH)

annotations/FRACAS_1premise_yesno/1h.yarn.json
annotations/FRACAS_1premise_yesno/1p.yarn.json
annotations/FRACAS_1premise_yesno/5h.yarn.json
annotations/FRACAS_1premise_yesno/5p.yarn.json
annotations/FRACAS_1premise_yesno/5p-2.yarn.json
annotations/FRACAS_1premise_yesno/6p.yarn.json
annotations/FRACAS_1premise_yesno/6h.yarn.json
annotations/FRACAS_1premise_yesno/7h.yarn.json
annotations/FRACAS_1premise_yesno/7p.yarn.json
annotations/FRACAS_1premise_yesno/8p.yarn.json
annotations/FRACAS_1premise_yesno/8h.yarn.json
annotations/FRACAS_1premise_yesno/9h.yarn.json
annotations/FRACAS_1premise_yesno/9p.yarn.json
annotations/FRACAS_1premise_yesno/10h.yarn.json
annotations/FRACAS_1premise_yesno/10p.yarn.json
annotations/FRACAS_1premise_yesno/15p.yarn.json
annotations/FRACAS_1premise_yesno/15h.yarn.json
annotations/FRACAS_1premise_yesno/17p.yarn.json
annotations/FRACAS_1premise_yesno/17h.yarn.json
annotations/FRACAS_1premise_yesno/23p.yarn.json
annotations/FRACAS_1premise_yesno/23h.yarn.json
ann

In [59]:
yarn2fol(corpus)


Processing: annotations/FRACAS_1premise_yesno/1p.yarn.json
annotations/FRACAS_1premise_yesno/1p.yarn.json
premise : An Italian became the world's greatest tenor.
∃t. ( tenor(t) ∧ ARG1(GREAT-02,t) ∧ C(t)
 ∧ (∃s1. ( S(s1)
 ∧ (∃p. ( person(p) ∧ mod(p,COUNTRY) ∧ C(p)
 ∧ (∃b. ( become-01(b) ∧ ARG1(b,p) ∧ ARG2(b,t) ∧ b≺NOW
 ∧ (∃w. ( world(w) ∧ C(w)
 )) )) )) )) )
---
∃t. ( tenor(t) ∧ ARG1(GREAT-02,t) ∧ C(t)
 ∧ (∃s1. ( S(s1)
 ∧ (∃w. ( world(w) ∧ C(w)
 )) )) )
---
∃t. ( tenor(t) ∧ ARG1(GREAT-02,t) ∧ C(t)
 ∧ (∃s1. ( S(s1)
 ∧ (∃p. ( person(p) ∧ mod(p,COUNTRY) ∧ C(p)
 ∧ (∃w. ( world(w) ∧ C(w)
 )) )) )) )
---
∃t. ( tenor(t) ∧ ARG1(GREAT-02,t) ∧ C(t)
 ∧ (∃s1. ( S(s1)
 ∧ (∃w. ( world(w) ∧ C(w)
 ∧ (∃p. ( person(p) ∧ mod(p,COUNTRY) ∧ C(p)
 ∧ (∃b. ( become-01(b) ∧ ARG1(b,p) ∧ ARG2(b,t) ∧ b≺NOW
 )) )) )) )) )
---
∃s1. ( S(s1)
 ∧ (∃p. ( person(p) ∧ mod(p,COUNTRY) ∧ C(p)
 ∧ (∃t. ( tenor(t) ∧ ARG1(GREAT-02,t) ∧ C(t)
 ∧ (∃w. ( world(w) ∧ C(w)
 ∧ (∃b. ( become-01(b) ∧ ARG1(b,p) ∧ ARG2(b,t) ∧ b≺NOW
 )) )) ))

In [54]:
yarn2fol(load_yarn('annotations/1.yarn.json'))

annotations/1.yarn.json

Processing: annotations/1.yarn.json
[]
annotations/1.yarn.json
None : Every young boy called John definitely doesn't want a large dog to bark.


# Vampire

In [35]:
def conj_tptp(parts):
    parts = [p for p in parts if p and p.strip()]
    return " & ".join(parts)

def wrap_quant_tptp(q, var, head, relations, body, connective):
    rel = conj_tptp(relations)
    head_part = f"{head}({var})"

    if rel:
        left = f"{head_part} & {rel}"
    else:
        left = head_part

    return f"{q} [{var}] : ( {left} {connective} ( {body} ) )"

def clean_formula_tptp(formula):
    clean_formula = formula.replace(" & (  )", "").replace("-", "_")
    return clean_formula

In [36]:
def interpret_vampire(root, temp_variable):
    
    if root is None:
        return ""
    
    if root["type"] == "∃_s" or root["type"] == "Q_exists":
        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )

    if root["type"] == "Q_forall":
        return wrap_quant_tptp(
            "!",
            root["variable"],
            root["tar_label"],
            root["relations"],
            interpret_vampire(root["child"], temp_variable),
            "=>"
        )

    if root["type"] == "Q_present":
        temp = f"{root['variable']}_O_{temp_variable}"

        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"] + [temp],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )

    if root["type"] == "Q_past":
        temp = f"before({root['variable']},{temp_variable})"

        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"] + [temp],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )
    
    if root["type"] == "Q_future":
        temp = f"before({temp_variable},{root['variable']})"

        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"] + [temp],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )
    
    if root["type"] == "Q_neg":
        return f"~( {interpret_vampire(root['child'], temp_variable)} )"

In [37]:
print(clean_formula_tptp(interpret_vampire(T_all[0], "NOW")))

NameError: name 'T_all' is not defined